# Praktikum 6: Apache Spark

**Mata Kuliah:** Praktikum Big Data
**Nama:** Arkan Ubaidillah Warman
**NIM:** 2411537001
**Program Studi:** Informatika, Fakultas Teknologi Informasi, Universitas Andalas

Notebook ini mengikuti langkah pada Modul Praktikum 6: persiapan PySpark dan SparkSession (D), RDD (E.1), DataFrame (E.2), agregasi dengan DataFrame API (E.3), Spark SQL (E.4), dan eksperimen lazy evaluation (F). Tugas Praktikum 6 ada di notebook terpisah.

## D. Persiapan Praktikum

**Langkah 3.** Instal PySpark.

In [1]:
!pip install pyspark

**Langkah 4.** Import library dan inisialisasi SparkSession. `appName()` memberi nama aplikasi, `master("local[*]")` menjalankan Spark secara lokal dengan semua core CPU, dan `getOrCreate()` membuat sesi baru atau memakai sesi yang sudah ada.

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PraktikumBigData") \
    .master("local[*]") \
    .getOrCreate()

print("Versi Spark:", spark.version)

Versi Spark: 4.3.0


**Langkah 5.** Periksa konfigurasi Spark.

In [3]:
print("Spark Application Name:",
      spark.sparkContext.appName)

print("Spark Master:",
      spark.sparkContext.master)

Spark Application Name: PraktikumBigData
Spark Master: local[*]


Nama aplikasi `PraktikumBigData` dan master `local[*]` sesuai dengan hasil yang diharapkan modul.

## E.1. Mengenal RDD pada Apache Spark

**Langkah 1.** Membuat RDD. `parallelize()` mengubah list Python menjadi RDD yang bisa diproses secara terdistribusi.

In [4]:
data = [10, 20, 30, 40, 50,
        60, 70, 80, 90, 100]

rdd = spark.sparkContext.parallelize(data)

print("Jumlah elemen:", rdd.count())

Jumlah elemen: 10


**Langkah 2.** Melihat isi RDD.

In [5]:
print(rdd.collect())

[10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


**Pertanyaan: Apa fungsi collect()? Mengapa operasi ini termasuk action?**

`collect()` mengambil seluruh elemen RDD dari semua partisi lalu mengirimnya ke driver sebagai list Python. Operasi ini termasuk action karena memaksa Spark benar-benar menjalankan pemrosesan dan mengembalikan nilai ke program, bukan sekadar membuat RDD baru.

**Langkah 3.** Transformation `map()`: kalikan setiap elemen dengan 2.

In [6]:
rdd_hasil = rdd.map(lambda x: x * 2)

print(rdd_hasil.collect())

[20, 40, 60, 80, 100, 120, 140, 160, 180, 200]


**Langkah 4.** Transformation `filter()`: ambil elemen yang lebih besar dari 50.

In [7]:
rdd_filter = rdd.filter(lambda x: x > 50)

print(rdd_filter.collect())

[60, 70, 80, 90, 100]


**Langkah 5.** Menggabungkan beberapa transformation: ambil bilangan genap, lalu kalikan dengan 3.

In [8]:
rdd_genap = rdd.filter(lambda x: x % 2 == 0)

rdd_kali = rdd_genap.map(lambda x: x * 3)

print(rdd_kali.collect())

[30, 60, 90, 120, 150, 180, 210, 240, 270, 300]


**Pertanyaan E.1**

1. **Perbedaan transformation dan action.** Transformation seperti `map()` dan `filter()` menghasilkan RDD baru dan hanya dicatat sebagai rencana. Action seperti `collect()` dan `count()` menjalankan rencana itu lalu mengembalikan hasil ke driver.
2. **Mengapa map() dan filter() tidak langsung dieksekusi.** Spark memakai lazy evaluation. Kedua operasi itu hanya ditambahkan ke lineage RDD, sehingga Spark bisa menyusun seluruh rangkaian langkah dulu dan menjalankannya sekaligus saat action dipanggil.
3. **Jika collect() tidak dipanggil.** `rdd_kali` tetap berupa rencana. Tidak ada data yang diproses dan tidak ada hasil yang muncul.
4. **Jika data sangat besar dan seluruh hasil collect() dikirim ke driver.** Driver bisa kehabisan memori (OutOfMemoryError) karena semua data dari banyak executor ditampung di satu mesin. Untuk data besar lebih aman memakai `take(n)`, `show()`, atau menyimpan hasil ke file.

## E.2. Membuat dan Mengolah DataFrame

**Langkah 1.** Membuat dataset transaksi penjualan.

In [9]:
data_transaksi = [
    (1, "Laptop", "Elektronik", 7500000, 2),
    (2, "Mouse", "Aksesoris", 150000, 5),
    (3, "Keyboard", "Aksesoris", 300000, 3),
    (4, "Monitor", "Elektronik", 2500000, 2),
    (5, "Flashdisk", "Aksesoris", 80000, 10),
    (6, "Printer", "Elektronik", 1800000, 1),
    (7, "Headset", "Aksesoris", 250000, 4),
    (8, "Laptop", "Elektronik", 7500000, 1),
    (9, "Mouse", "Aksesoris", 150000, 8),
    (10, "Monitor", "Elektronik", 2500000, 3)
]

kolom = [
    "id_transaksi",
    "produk",
    "kategori",
    "harga",
    "jumlah"
]

df = spark.createDataFrame(data_transaksi, kolom)

**Langkah 2.** Menampilkan data.

In [10]:
df.show()

+------------+---------+----------+-------+------+
|id_transaksi|   produk|  kategori|  harga|jumlah|
+------------+---------+----------+-------+------+
|           1|   Laptop|Elektronik|7500000|     2|
|           2|    Mouse| Aksesoris| 150000|     5|
|           3| Keyboard| Aksesoris| 300000|     3|
|           4|  Monitor|Elektronik|2500000|     2|
|           5|Flashdisk| Aksesoris|  80000|    10|
|           6|  Printer|Elektronik|1800000|     1|
|           7|  Headset| Aksesoris| 250000|     4|
|           8|   Laptop|Elektronik|7500000|     1|
|           9|    Mouse| Aksesoris| 150000|     8|
|          10|  Monitor|Elektronik|2500000|     3|
+------------+---------+----------+-------+------+



**Langkah 3.** Melihat struktur DataFrame.

In [11]:
df.printSchema()

root
 |-- id_transaksi: long (nullable = true)
 |-- produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- jumlah: long (nullable = true)



Spark menebak tipe kolom dari data Python: angka bulat menjadi `long` dan teks menjadi `string`. Semua kolom bertanda `nullable = true`.

**Langkah 4.** Menghitung jumlah baris.

In [12]:
print("Jumlah transaksi:", df.count())

Jumlah transaksi: 10


**Langkah 5.** Memilih kolom produk, kategori, dan harga.

In [13]:
df.select(
    "produk",
    "kategori",
    "harga"
).show()

+---------+----------+-------+
|   produk|  kategori|  harga|
+---------+----------+-------+
|   Laptop|Elektronik|7500000|
|    Mouse| Aksesoris| 150000|
| Keyboard| Aksesoris| 300000|
|  Monitor|Elektronik|2500000|
|Flashdisk| Aksesoris|  80000|
|  Printer|Elektronik|1800000|
|  Headset| Aksesoris| 250000|
|   Laptop|Elektronik|7500000|
|    Mouse| Aksesoris| 150000|
|  Monitor|Elektronik|2500000|
+---------+----------+-------+



**Langkah 6.** Filtering: transaksi dengan harga produk lebih dari Rp1.000.000.

In [14]:
df.filter(
    df.harga > 1000000
).show()

+------------+-------+----------+-------+------+
|id_transaksi| produk|  kategori|  harga|jumlah|
+------------+-------+----------+-------+------+
|           1| Laptop|Elektronik|7500000|     2|
|           4|Monitor|Elektronik|2500000|     2|
|           6|Printer|Elektronik|1800000|     1|
|           8| Laptop|Elektronik|7500000|     1|
|          10|Monitor|Elektronik|2500000|     3|
+------------+-------+----------+-------+------+



**Langkah 7.** Membuat kolom baru `total` = harga x jumlah.

In [15]:
from pyspark.sql.functions import col

df_total = df.withColumn(
    "total",
    col("harga") * col("jumlah")
)

df_total.show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
+------------+---------+----------+-------+------+--------+



**Langkah 8.** Mengurutkan transaksi berdasarkan total dari terbesar ke terkecil.

In [16]:
df_total.orderBy(
    col("total").desc()
).show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
+------------+---------+----------+-------+------+--------+



## E.3. Agregasi Data Menggunakan DataFrame API

**Langkah 1.** Total pendapatan setiap kategori.

In [17]:
from pyspark.sql.functions import sum

df_total.groupBy("kategori") \
    .agg(
        sum("total").alias("total_pendapatan")
    ).show()

+----------+----------------+
|  kategori|total_pendapatan|
+----------+----------------+
|Elektronik|        36800000|
| Aksesoris|         4650000|
+----------+----------------+



**Langkah 2.** Jumlah transaksi setiap kategori.

In [18]:
from pyspark.sql.functions import count

df_total.groupBy("kategori") \
    .agg(
        count("*").alias("jumlah_transaksi")
    ).show()

+----------+----------------+
|  kategori|jumlah_transaksi|
+----------+----------------+
|Elektronik|               5|
| Aksesoris|               5|
+----------+----------------+



**Langkah 3.** Rata-rata nilai transaksi setiap kategori.

In [19]:
from pyspark.sql.functions import avg

df_total.groupBy("kategori") \
    .agg(
        avg("total").alias("rata_rata_transaksi")
    ).show()

+----------+-------------------+
|  kategori|rata_rata_transaksi|
+----------+-------------------+
|Elektronik|          7360000.0|
| Aksesoris|           930000.0|
+----------+-------------------+



**Langkah 4.** Statistik keseluruhan.

In [20]:
df_total.select(
    sum("total").alias("total_penjualan"),
    avg("total").alias("rata_rata_penjualan"),
    count("*").alias("jumlah_transaksi")
).show()

+---------------+-------------------+----------------+
|total_penjualan|rata_rata_penjualan|jumlah_transaksi|
+---------------+-------------------+----------------+
|       41450000|          4145000.0|              10|
+---------------+-------------------+----------------+



**Pertanyaan analisis E.3**

1. **Kategori dengan pendapatan terbesar** adalah Elektronik dengan Rp36.800.000, jauh di atas Aksesoris yang hanya Rp4.650.000, padahal jumlah transaksinya sama-sama 5.
2. **Total pendapatan seluruh transaksi** adalah Rp41.450.000 dari 10 transaksi, dengan rata-rata Rp4.145.000 per transaksi.
3. **Mengapa groupBy() diperlukan.** Tanpa `groupBy()`, agregasi hanya menghasilkan satu angka untuk seluruh data. `groupBy()` memecah data per nilai kolom, misalnya per kategori, sehingga kita bisa membandingkan kelompok.
4. **Perbedaan sum(), count(), dan avg().** `sum()` menjumlahkan nilai kolom, `count()` menghitung banyaknya baris, dan `avg()` menghitung rata-rata, yaitu hasil `sum()` dibagi `count()`.

## E.4. Pemrosesan Data Menggunakan Spark SQL

**Langkah 1.** Membuat temporary view bernama `transaksi`.

In [21]:
df_total.createOrReplaceTempView(
    "transaksi"
)

**Langkah 2.** Menampilkan seluruh data.

In [22]:
spark.sql("""
    SELECT *
    FROM transaksi
""").show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
+------------+---------+----------+-------+------+--------+



**Langkah 3.** Filtering dengan SQL: transaksi dengan total lebih dari 1.000.000.

In [23]:
spark.sql("""
    SELECT produk, kategori, harga, jumlah, total
    FROM transaksi
    WHERE total > 1000000
""").show()

+-------+----------+-------+------+--------+
| produk|  kategori|  harga|jumlah|   total|
+-------+----------+-------+------+--------+
| Laptop|Elektronik|7500000|     2|15000000|
|Monitor|Elektronik|2500000|     2| 5000000|
|Printer|Elektronik|1800000|     1| 1800000|
| Laptop|Elektronik|7500000|     1| 7500000|
|  Mouse| Aksesoris| 150000|     8| 1200000|
|Monitor|Elektronik|2500000|     3| 7500000|
+-------+----------+-------+------+--------+



**Langkah 4.** Agregasi dengan SQL.

In [24]:
spark.sql("""
    SELECT kategori,
           SUM(total) AS total_pendapatan,
           COUNT(*) AS jumlah_transaksi,
           AVG(total) AS rata_rata_transaksi
    FROM transaksi
    GROUP BY kategori
""").show()

+----------+----------------+----------------+-------------------+
|  kategori|total_pendapatan|jumlah_transaksi|rata_rata_transaksi|
+----------+----------------+----------------+-------------------+
|Elektronik|        36800000|               5|          7360000.0|
| Aksesoris|         4650000|               5|           930000.0|
+----------+----------------+----------------+-------------------+



**Langkah 5.** Mengurutkan hasil query.

In [25]:
spark.sql("""
    SELECT produk,
           SUM(total) AS total_penjualan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_penjualan DESC
""").show()

+---------+---------------+
|   produk|total_penjualan|
+---------+---------------+
|   Laptop|       22500000|
|  Monitor|       12500000|
|    Mouse|        1950000|
|  Printer|        1800000|
|  Headset|        1000000|
| Keyboard|         900000|
|Flashdisk|         800000|
+---------+---------------+



**Pertanyaan: Bandingkan hasil Spark SQL dengan DataFrame API pada E.3. Apakah hasilnya sama?**

Sama. Query agregasi SQL menghasilkan total pendapatan Elektronik 36.800.000 dan Aksesoris 4.650.000, jumlah transaksi 5 dan 5, serta rata-rata 7.360.000 dan 930.000, persis seperti Langkah 1 sampai 3 pada E.3. Bedanya hanya bentuk: SQL menampilkan ketiga agregasi dalam satu tabel, sedangkan E.3 memakai tiga perintah terpisah. Keduanya dijalankan oleh mesin yang sama (Catalyst optimizer), jadi hasilnya pasti sama.

## F. Eksperimen Lazy Evaluation

Sel pertama hanya mendefinisikan transformation.

In [26]:
df_filter = df.filter(
    df.harga > 1000000
)

print("Transformation selesai")

Transformation selesai


Sel di atas hanya mencetak "Transformation selesai" tanpa menampilkan data. Data baru muncul setelah action `show()` dipanggil.

In [27]:
df_filter.show()

+------------+-------+----------+-------+------+
|id_transaksi| produk|  kategori|  harga|jumlah|
+------------+-------+----------+-------+------+
|           1| Laptop|Elektronik|7500000|     2|
|           4|Monitor|Elektronik|2500000|     2|
|           6|Printer|Elektronik|1800000|     1|
|           8| Laptop|Elektronik|7500000|     1|
|          10|Monitor|Elektronik|2500000|     3|
+------------+-------+----------+-------+------+



Rencana eksekusi Spark untuk `df_filter`:

In [28]:
df_filter.explain()

== Physical Plan ==
*(1) Filter (isnotnull(harga#3L) AND (harga#3L > 1000000))
+- *(1) Scan ExistingRDD[id_transaksi#0L,produk#1,kategori#2,harga#3L,jumlah#4L]




**Pertanyaan F**

1. **Mengapa Spark menggunakan lazy evaluation.** Spark perlu melihat seluruh rangkaian transformation sebelum menjalankannya, supaya bisa menyusun rencana eksekusi yang paling efisien dan tidak memproses data yang tidak dibutuhkan.
2. **Keuntungan lazy evaluation untuk Big Data.** Beberapa transformation bisa digabung dalam satu tahap tanpa menulis hasil antara ke disk, filter bisa didorong sedekat mungkin ke sumber data, dan hanya kolom serta baris yang diperlukan yang dibaca. Pada data berukuran besar, penghematan ini berarti waktu dan memori yang jauh lebih kecil.
3. **Fungsi execution plan.** `explain()` menunjukkan langkah yang benar-benar akan dijalankan Spark. Pada output di atas, rencana fisik (Physical Plan) berisi `Scan ExistingRDD` lalu `Filter` dengan kondisi `isnotnull(harga#3L) AND (harga#3L > 1000000)`, artinya Spark membaca data dari RDD sumber lalu menyaring baris. Tanda `*(1)` menunjukkan kedua langkah digabung dalam satu tahap kode yang sama (whole-stage codegen). Rencana ini berguna untuk memeriksa apakah query sudah efisien.
4. **Perbedaan mendefinisikan transformation dan mengeksekusi action.** Mendefinisikan `df.filter(...)` hanya membuat rencana dan selesai seketika, seperti terlihat dari cetakan "Transformation selesai" tanpa data. Mengeksekusi `df_filter.show()` menjalankan rencana itu pada data dan menampilkan lima baris hasilnya.